In [123]:
import re, json, os
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv
import time

In [124]:
# Load .env from the parent directory of the current working directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
load_dotenv(dotenv_path=os.path.join(parent_dir, ".env"))


True

In [125]:
# Make sure the environment variable is loaded
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables")

In [126]:
# Initialize API client
client = OpenAI()

In [127]:
# ----- STEP 1: Load raw JSON -----
with open("../data/springfield_locations.json", "r", encoding="utf-8") as f:
    locations = json.load(f)

In [128]:
# ----- STEP 2: Basic normalization -----
def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)           # collapse whitespace
    text = re.sub(r"[^a-z0-9\s.,!?'\-]", "", text)  # strip weird chars
    return text.strip()


In [129]:
# ----- STEP 3: ChatGPT cleaner -----

def improve_entry(name, loc_type, raw_desc):
    """
    Use GPT function-calling to return a structured Springfield location entry.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Springfield Encyclopedia editor. "
                    "You will receive scraped descriptions of Simpsons locations. "
                    "Your job is to return a structured JSON object with canonical fields. "
                    "Write the description as 1–5 full paragraphs in a detailed, neutral, encyclopedia style. "
                    "The show is called 'The Simpsons' and so it is very important to explain the relevance of the location to the various members of the Simpsons family and to other characters."
                    "Expand on purpose, history, appearance, canonical characters, key activities, and notable events. "
                    "Keep only Springfield facts. Remove episode lists, trivia, or meta-commentary. "
                    "Infer missing values (e.g., location_type). "
                    "Do not start the description with the location name. "
                    "Do not output anything except valid JSON per the schema."
                )
            },
            {
                "role": "user",
                "content": f"Name: {name}\nType: {loc_type}\nRaw: {raw_desc}"
            }
        ],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "store_location_entry",
                    "description": "Store a cleaned Springfield location entry",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location_name": {"type": "string"},
                            "location_type": {"type": "string"},
                            "description": {"type": "string"},
                            "canonical_characters": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "key_activities": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "notable_events": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "relevance_to_the_simpson_family": {
                                "type": "array",
                                "items": {"type": "string"}
                            }
                        },
                        "required": [
                            "location_name",
                            "location_type",
                            "description",
                            "canonical_characters",
                            "key_activities",
                            "notable_events",
                            "relevance_to_the_simpson_family"
                        ]
                    }
                }
            }
        ],
        tool_choice={"type": "function", "function": {"name": "store_location_entry"}},
    )

    # Extract structured JSON from tool call
    args = response.choices[0].message.tool_calls[0].function.arguments
    return json.loads(args)


In [130]:
# ----- STEP 4: Deduplication -----
def deduplicate(data):
    seen = set()
    deduped = []
    for loc in data:
        key = (loc["location_name"].lower(), loc["description"].lower())
        if key not in seen:
            deduped.append(loc)
            seen.add(key)
    return deduped

In [131]:

# ----- STEP 5: Process dataset -----
cleaned_locations = []
for loc in tqdm(locations): 
    
    name = loc.get("location_name", "").strip()
    loc_type = loc.get("location_type", "").strip()
    raw_desc = normalize(loc.get("location_description", ""))

    improved = improve_entry(name, loc_type, raw_desc)

    # append full structured result
    cleaned_locations.append(improved)
    
    time.sleep(1)  # pause 1 sec between requests

cleaned_locations = deduplicate(cleaned_locations)

100%|██████████| 1155/1155 [2:17:45<00:00,  7.16s/it] 


In [132]:

# ----- STEP 6: Save cleaned JSON & CSV -----
json_path = "../data/springfield_locations_cleaned.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_locations, f, indent=2, ensure_ascii=False)

csv_path = "../data/springfield_locations_cleaned.csv"
pd.DataFrame(cleaned_locations).to_csv(csv_path, index=False)

print("✅ Springfield dataset cleaned & enriched (JSON + CSV saved)!")

✅ Springfield dataset cleaned & enriched (JSON + CSV saved)!
